In [1]:
import os
import pandas as pd
from clickhouse_driver import Client
import math
import datetime


client = Client(host='clickhouse', port=9000)

DUMP_DIR = '/home/jovyan/work/dumps'
os.makedirs(DUMP_DIR, exist_ok=True)

result = client.execute('SELECT version()')
print(f'ClickHouse version: {result[0][0]}')

ClickHouse version: 23.8.16.16


/tmp/ipykernel_3333/2061428621.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
def insert_batches(query, rows, batch_size=1000):
    total = len(rows)
    batches = math.ceil(total / batch_size)
    for i in range(batches):
        batch = rows[i * batch_size:(i + 1) * batch_size]
        client.execute(query, batch)
    print(f'  Вставлено: {total:,} строк ({batches} батчей)')

In [3]:
df = pd.read_csv(f'{DUMP_DIR}/currency_rates.csv')

rows = [
    (datetime.date.fromisoformat(str(r.dt)), r.currency, float(r.rate_to_rub), int(r.nominal))
    for r in df.itertuples()
]

client.execute('TRUNCATE TABLE currency_rates')
insert_batches(
    'INSERT INTO currency_rates (dt, currency, rate_to_rub, nominal) VALUES',
    rows
)
print('currency_rates восстановлена')

  Вставлено: 15,033 строк (16 батчей)
currency_rates восстановлена


In [10]:
df = pd.read_csv(f'{DUMP_DIR}/transactions_valid.csv')

rows = [
    (
        int(r.transaction_id), int(r.account_id), str(r.account_level),
        Decimal(str(r.daily_limit_rub)), Decimal(str(r.monthly_limit_rub)),
        Decimal(str(r.amount)), float(r.amount_rub), Decimal(str(r.account_balance)), str(r.currency),
        str(r.transaction_type), str(r.merchant_category), str(r.category_name),
        int(r.risk_score), int(r.is_online),
        str(r.country_code), str(r.country_name), str(r.region), int(r.risk_level),
        datetime.fromisoformat(str(r.timestamp)),
        date.fromisoformat(str(r.transaction_date)),
        datetime.fromisoformat(str(r.ingested_at))
    )
    for r in df.itertuples()
]

client.execute('TRUNCATE TABLE transactions_valid')
insert_batches(
    '''
    INSERT INTO transactions_valid (
        transaction_id, account_id, account_level,
        daily_limit_rub, monthly_limit_rub,
        amount, amount_rub, account_balance, currency,
        transaction_type, merchant_category, category_name,
        risk_score, is_online,
        country_code, country_name, region, risk_level,
        timestamp, transaction_date, ingested_at
    ) VALUES
    ''',
    rows
)
print('transactions_valid восстановлена')

  Вставлено: 1,613 строк (2 батчей)
transactions_valid восстановлена


In [11]:
df = pd.read_csv(f'{DUMP_DIR}/transactions_invalid.csv')

rows = [
    (
        int(r.transaction_id), int(r.account_id), str(r.account_level),
        Decimal(str(r.amount)), Decimal(str(r.account_balance)), str(r.currency),
        str(r.transaction_type), str(r.merchant_category),
        str(r.country_code),
        datetime.fromisoformat(str(r.timestamp)) if pd.notna(r.timestamp) else None,
        date.fromisoformat(str(r.transaction_date)) if pd.notna(r.transaction_date) else None,
        datetime.fromisoformat(str(r.ingested_at))
    )
    for r in df.itertuples()
]

client.execute('TRUNCATE TABLE transactions_invalid')
insert_batches(
    '''
    INSERT INTO transactions_invalid (
        transaction_id, account_id, account_level,
        amount, account_balance, currency,
        transaction_type, merchant_category,
        country_code, timestamp, transaction_date, ingested_at
    ) VALUES
    ''',
    rows
)
print('transactions_invalid восстановлена')

  Вставлено: 108 строк (1 батчей)
transactions_invalid восстановлена


In [12]:
print('Проверяем таблицы')
for table in ['currency_rates', 'transactions_valid', 'transactions_invalid']:
    result = client.execute(f'SELECT count() FROM {table}')
    print(f'{table}: {result[0][0]:,} строк')

Проверяем таблицы
currency_rates: 15,017 строк
transactions_valid: 1,613 строк
transactions_invalid: 106 строк
